# Szkic pracy inżynierskiej
## Temat pracy: Opracowanie hybrydowego systemu komunikacji optyczno-radiowej (VLC/RF) dla zapewnienia łączności obiektów IoT.
### Autor: Ivan Ovsiienko
#### Wersja: 0.2

---

### Opis Architektury Systemu
#### Architektura hybrydowego systemu komunikacji optyczno-radiowej (VLC/RF) dla zapewnienia łączności obiektów IoT.

##### 1. Topologia i Koncepcja

##### 1.1 Koncepcja ogólna

Projekt zakłada realizację hybrydowego systemu transmisji danych. System zaprojektowano w topologii MasterSlave (Gwiazda), gdzie centralnym punktem (Master) jest zródło oświetlenia głównego (Lampa/Taśma LED), a punktami końcowymi (Slave) są urządzenia IoT, obejmujące zarówno sensory (np. temperatury), jak i elementy wykonawcze (np. inteligentne gniazdka, sterowniki rolet).
System wykorzystuje dywersyfikację kanałów komunikacyjnych w celu optymalizacji zużycia energii, niezawodnoscie oraz koszt produkcji.

 - Łączę w dół (Downlink): Transmisja od Lampy do Urzadzenia końcowego odbywa się poprzez modulację swiatła widzialnego (VLC).
   Jest  to kanał typu Broadcast (Jeden do Wielu) - lampa oswietla cały pokój, wysyłając dane synchronizacyjne, geolokalizację i/lub komendy.
   Mimo tego, Lampa ma kanał rezerwowy na wypadek wylączenia.

 - Łącze w górę (Uplink): Komunikacja zwrotna od Slave do Master realizowana jest w paśmie radiowym (ISM 2.4 GHz) do przesyłania telemetrii z czujników, co zamyka pętlę sterowania i elimunuje problem widoczności bezpośredniej (dla kanału zwrotnego).

##### 1.2 Opis Komponentów Sprzętowych

1. Stacja Bazowa - Inteligentna Lampa (Master Node)

Urządzenia zasiłane z sieci, pełniace funkcję bramy sieciowej (Gateway).
- Moduł oswietleniowy: Taśma LED 12V/Lampa LED 5V sterowana tranzystorem MOSFET.
- Sterowanie VLC: Mikrokontroler generuje sygnał PWM z kodowaniem Manchester lub Wypełnienie 5-8b/10b, zapewniając transmisję danych bez widocznego efektu migotania (flicker-free). **_DO EDYCJI I RESEARCH_**
- Odbiornik RF: Moduł radiowy 2.4Ghz do odbierania danych z urządzeń.
- Nadajnik RF: Moduł radiowy 2.4Ghz do przesyłania danych do urządzeń.
- Łączność z Chmurą i internetem: Moduł Wi-Fi (np. ESP32) przesyłający zebrane dane do sieci domowej (np. MOTT/Home Assistant) oraz pobiera aktualny czas sieci. **_DODATEK ROZWOJU. DO ZASTANOWIENIA_**

2. Węzeł Końcowy — Urządzenie loT (End Node:Sensor/Actuator)

Urządzenie pełniące rolę klienta w topologii gwiazdy. W zależności od funkcji, może być zasilane bateryjnie (np. czujniki) lub sieciowo (np. gniazdka 230V).
- Typy urządzeń:
  1. Sensory (Input): Pasywne zbieranie danych środowiskowych (temperatura, wilgotność, ruch). Priorytet: oszczędzanie energii.
  2. Aktuatory (Output): Elementy wykonawcze, takie jak Smart Plugs (Inteligentne Gniazdka), przekaźniki czy sterowniki LED. Priorytet: niezawodność sterowania i krótki czas reakcji (latencja).
- Odbiornik Optyczny (VLC/OWS): Służy do odbierania poleceń sterujących (np. "Włącz zasilanie") oraz danych geolokalizacyjnych bezpośrednio z oświetlenia.
- Transceiver RF 2.4Ghz:
    - Dla czujników: Wysyłanie pomiarów, aktualizacja oprogramowania układowego.
    - Dla gniazdek/aktuatorów: Odbieranie komend, gdy światło jest wyłączone lub receiver jest zasłonięty, oraz wysyłanie potwierdzeń wykonania akcji (ACK) do Lampy-Bramy.

##### Q&A (Analiza Krytyczna)

- Pytanie 1: Dlaczego nie zastosowano Wi-Fi? (Argument dla Czujników i Gniazdek)
    Wątpliwość: Gniazdka mają zasilanie z sieci, więc oszczędzanie baterii nie jest argumentem. Dlaczego nie użyć w nich Wi-Fi? Uzasadnienie:
  - Skalowalność i problem saturacji widma (Spectrum Congestion): W nowoczesnej instalacji Smart Home liczba węzłów sieciowych (gniazdka, przełączniki, sterowniki LED) może sięgać kilkudziesięciu urządzeń na jeden lokal. Podłączenie wszystkich elementów wykonawczych do standardowego routera Wi-Fi (2.4 GHz) prowadzi do drastycznej degradacji parametrów łącza internetowego, objawiającej się wzrostem opóźnień (latency) oraz zwiększoną liczbą kolizji pakietów (CSMA/CA collisions). Problem ten eskaluje w budownictwie wielorodzinnym (blokach mieszkalnych). Ze względu na przenikalność fal radiowych przez ściany i stropy, sieci Wi-Fi sąsiadujących mieszkań nakładają się na siebie. Przy dużym zagęszczeniu lokali, gdzie każdy użytkownik posiadałby kilkadziesiąt urządzeń loT, następuje nasycenie eteru i gwałtowny wzrost interferencji współkanałowych (Co-channel Interference).
  - Izolacja fizyczna: Sygnał optyczny (VLC) jest naturalnie ograniczony barierami architektonicznymi (ściany), co tworzy fizycznie odseparowane mikro-komórki (Micro-cells) i zapobiega zakłócaniu systemów sąsiednich.
  - Energoefektywność: jest opisane w akapicie poniżej. 

- Pytanie 2: Jaką przewagę daje sterowanie światłem (VLC) dla Gniazdka?
  Wątpliwość: Czy nie prościej sterować gniazdkiem tylko przez radio? Uzasadnienie Inżynierskie:
  - Geolokalizacja (Indoor Positioning): To kluczowa przewaga VLC. Fale radiowe przenikają przez ściany, co utrudnia określenie, w którym pokoju znajduje się czujnik. Światło jest ograniczone ścianami. Jeśli czujnik odbiera sygnał ID lampy, system ma 100% pewności, w którym pomieszczeniu się on znajduje. Pozwala to na automatyczną konfigurację (Zero-touch
provisioning).
  - Broadcast bez Kolizji: Lampa może wysłać komendę do 100 czujników w pokoju jednocześnie (np. "zsynchronizuj czas"). W przypadku radia mogłoby to doprowadzić do kolizji pakietów.
  - Bezpieczeństwo Fizyczne: Sygnał VLC nie może zostać podsłuchany ani zakłócony z zewnątrz budynku (spoza pomieszczenia), co czyni go idealnym medium do wymiany kluczy szyfrujących.
  - Energooszczędność: Wykorzystanie włączonego odbiornika RF pobiera znaczną ilość prądu, co drastycznie skraca czas życia baterii. Dlatego uzywamy przerwania snu węzła przy pomocy swiatła.

- Pytanie 3: Co ze sterowaniem gniazdkiem/czujnikiem, gdy jest ciemno?
    - W tym przypadku system działa w trybie RF-First. Komenda "Włącz gniazdko" wysyłana jest przez radio 2.4GHz z Lampy-Bramy. Radio w gniazdkach zasilanych sieciowo jest stale włączone (nie musi oszczędzać energii jak w czujnikach), co gwarantuje natychmiastową reakcję nawet w nocy. Mimo tego zostawiono było opcję rozwinięcia systemu z VLC do OWC(Optical Wireless Communication) dodaniem IR diod. Co sprawi niezawodność systemu nawet przy wyłączonym oswietleniu w nocy.  
- Pytanie 4: Dlaczego w kanale zwrotnym (Uplink) wybrano radio zamiast Podczerwieni (IR)?
Wątpliwość: Podczerwień też jest tania i optyczna, dlaczego z niej zrezygnowano? Uzasadnienie:
    - Problem Interferencji Słonecznej: Promieniowanie słoneczne zawiera silną składową podczerwoną, która może "oślepić" odbiornik IR na suficie w słoneczny dzień. Radio jest całkowicie odporne na warunki oświetleniowe.
    - Brak wymogu widoczności (NLOS): Czujnik może zostać zasłonięty (np. leżeć pod kanapą lub w kieszeni). Podczerwień by nie zadziałała. Radio 2.4GHz ma wystarczającą przenikalność, by zapewnić stabilne połączenie w obrębie jednego pomieszczenia.


##### Podsumowanie Celowości Projektu

Projekt nie ma na celu zastąpienia Wi-Fi, lecz uzupełnienie infrastruktury loT. Proponowane rozwiązanie eliminuje "wąskie gardła" współczesnych systemów Smart Home: problem częstej wymiany baterii w czujnikach oraz problem zatłoczenia sieci radiowych, oferując w zamian unikalną funkcję precyzyjnej lokalizacji wewnątrzbudynkowej.